# D3 · M3.2 — LangGraph 1.0: Stateful, Durable Workflows

**Worked side by side with the Concept HTML page, not read start to finish.** Read a short
concept section on the page, run the matching task below, then go back to the page for the next
concept — seven round trips, not one long lab.

**THE SITUATION:** M3.1 (earlier today) built an agent that reasons and calls tools — but it has
no memory between runs, no pause button, and no comeback after a crash. Today you wrap that same
kind of reasoning inside a LangGraph workflow that fixes all three: a Meridian UPI-transfer triage
workflow that pauses for a human's approval before a transfer that breaches the daily limit, and
resumes correctly — even after a real restart. **This is your Capstone Phase 3 artifact.**

**This notebook is fully working code — nothing to write.** Run each cell in order, read what
it prints, and follow the concept page's lab-marker cues to know when to come back here.

Concept page: `Day3/concept/D3_M3.2_LangGraph_Workflows_Concept.html`


## Setup

Loads your API key and the Day 2 Meridian article data this lab reuses (the same 40 articles
M3.1's `search_meridian_kb` used). Run this once before anything else.


In [1]:
# Run once. "Requirement already satisfied" is a good outcome.
%pip install -q python-dotenv langchain langchain-openai langgraph langgraph-checkpoint-sqlite


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import sqlite3
import time
import uuid
from pathlib import Path
from typing import Optional, TypedDict, Annotated
import operator

from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_openai import OpenAIEmbeddings
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt, Command


# Same walk-up .env search every Day 2/3 notebook uses -- one key file for the whole repo,
# wherever you open this from.
def find_env():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / ".env").is_file():
            return candidate / ".env"
        legacy = candidate / "Day1" / "labs" / "starter" / ".env"
        if legacy.is_file():
            return legacy
    return None


_env = find_env()
if _env:
    load_dotenv(_env)
    print(f"Loaded API key from: {_env}")
else:
    print("WARNING: no .env file found -- see SETUP.md Step 2.")

import os
assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY is not set. Set it before continuing -- never hard-code a key in this notebook."
)


# Same walk-up idea, but for a data file -- avoids the hardcoded parents[N] bug earlier Day 2
# notebooks had (see DELIVERY_PREREQS.md).
def find_repo_file(relative_path):
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        target = candidate / relative_path
        if target.is_file():
            return target
        target = candidate / "student_repo" / relative_path
        if target.is_file():
            return target
    raise FileNotFoundError(f"Could not find {relative_path} above {here}")


ARTICLES_PATH = find_repo_file(Path("Day2") / "data" / "D2_M2.3_meridian_articles.json")
with open(ARTICLES_PATH, "r", encoding="utf-8") as f:
    MERIDIAN_ARTICLES = json.load(f)
print(f"Loaded {len(MERIDIAN_ARTICLES)} Meridian articles from: {ARTICLES_PATH}")

model = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)

# IMPORTANT: the checkpoint database this lab creates is a real SQLite file. If you ever see a
# "disk I/O error" from SqliteSaver, you are very likely running this notebook from a
# cloud-synced folder (OneDrive/Dropbox/Google Drive) -- SQLite's file-locking needs a real local
# disk. Copy the repo to a local folder (not a synced one) if that happens.
print("Ready.")


Loaded API key from: C:\Users\Administrator\Training\EY_GenAI_3D\Day1\labs\starter\.env
Loaded 40 Meridian articles from: C:\Users\Administrator\Training\EY_GenAI_3D\Day2\data\D2_M2.3_meridian_articles.json
Ready.


## Foundation, on the page — then come back here for Task 1

Read **Why Can't This Morning's Agent Do This?** on the concept page, then read **Concept A** and
**Concept B** (the graph model, and branching on risk), then run the cells below — they build
the real Meridian triage graph and run it two ways, with no checkpointer yet.


In [3]:
# TASK 1a -- the shared state, and the first two nodes: triage (real extraction) and
# policy_check (reuses M3.1's exact search_meridian_kb approach over the same 40 articles).

class TriageState(TypedDict):
    customer_message: str
    customer_name: str
    account_type: str
    intent: str
    amount: Optional[float]
    kb_note: str
    decision: str
    # Annotated with operator.add: LangGraph ADDS new log entries to this list instead of
    # replacing it -- this is the "reducer" idea from Concept A. Every other field above uses
    # the default behaviour (a node's return value simply overwrites the old one).
    log: Annotated[list, operator.add]


class TriageExtraction(BaseModel):
    """Structured extraction of a Meridian customer request -- same field names as Day 1's
    M1.2 extractor (name, account_type, intent), plus an amount for this module's UPI scenario."""
    name: str = Field(description="Customer's name, mentioned or implied in the message")
    account_type: str = Field(description="e.g. 'savings', 'current'")
    intent: str = Field(description="e.g. 'upi_transfer', 'balance_inquiry', 'account_closure'")
    amount: float = Field(description="Rupee amount involved in the request; 0 if none is mentioned")


extractor_model = model.with_structured_output(TriageExtraction)


def triage_node(state: TriageState) -> dict:
    """Node 1. Reads only customer_message from state. Writes name/account_type/intent/amount --
    this is the 'extraction' skill from Day 1 M1.2 and Day 2 M2.1, reused as one node here."""
    extraction = extractor_model.invoke(state["customer_message"])
    return {
        "customer_name": extraction.name,
        "account_type": extraction.account_type,
        "intent": extraction.intent,
        "amount": extraction.amount,
        "log": [f"triage: extracted intent={extraction.intent}, amount={extraction.amount}"],
    }


# Embeds the same 40 Meridian articles M3.1 used, once, at import time.
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")
ARTICLE_TEXTS = [f"{a['title']}. {a['text']}" for a in MERIDIAN_ARTICLES]
ARTICLE_VECTORS = embeddings_model.embed_documents(ARTICLE_TEXTS)
print(f"Embedded {len(ARTICLE_VECTORS)} articles for policy_check_node to search.")


def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = sum(x * x for x in a) ** 0.5
    norm_b = sum(y * y for y in b) ** 0.5
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0


@tool
def search_meridian_kb(query: str) -> str:
    """Search Meridian Bank's internal knowledge base for a policy, fee, or procedure answer."""
    query_vector = embeddings_model.embed_query(query)
    scored = [
        (cosine_similarity(query_vector, vec), article)
        for vec, article in zip(ARTICLE_VECTORS, MERIDIAN_ARTICLES)
    ]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    best_score, best_article = scored[0]
    return f"[{best_article['id']}] {best_article['title']}. {best_article['text']}"


WIRE_LIMIT_QUERY = "UPI daily transaction limit"
DAILY_LIMIT = 100000.0  # matches the real KB article M3.1 already verified: "up to one lakh
                        # rupees (Rs 1,00,000) per day" -- this policy_check node fetches that
                        # SAME article live, it does not hardcode the number as a fact on faith.


def policy_check_node(state: TriageState) -> dict:
    """Node 2. Looks up Meridian's real UPI limit policy via search -- the 'retrieval' skill from
    Day 2, reused here as a graph node instead of an agent tool call."""
    note = search_meridian_kb.func(WIRE_LIMIT_QUERY)
    return {"kb_note": note, "log": ["policy_check: fetched UPI daily limit policy"]}


print("triage_node and policy_check_node defined.")


Embedded 40 articles for policy_check_node to search.
triage_node and policy_check_node defined.


In [4]:
# TASK 1b -- the conditional edge, the remaining nodes, and compiling WITHOUT a checkpointer
# yet (that's Concept C, next). This is a real langgraph.StateGraph -- state/nodes/edges,
# exactly as shown on the concept page.

def route_by_risk(state: TriageState) -> str:
    """A conditional edge is just a function that returns the name of the next node. Nothing
    fancier than an if/else that the graph follows."""
    if state["amount"] > DAILY_LIMIT:
        return "human_approval"
    return "auto_process"


def human_approval_node(state: TriageState) -> dict:
    """Node 3 (sensitive branch). interrupt() genuinely pauses the graph here -- see what
    happens below when we try to resume it without a checkpointer."""
    decision = interrupt({
        "question": f"Approve this Rs {state['amount']:,.0f} UPI transfer for {state['customer_name']}? "
                     f"It breaches today's Rs {DAILY_LIMIT:,.0f} limit.",
        "kb_note": state["kb_note"],
    })
    return {"decision": decision, "log": [f"human_approval: {decision}"]}


def auto_process_node(state: TriageState) -> dict:
    """Node 3 (safe branch). No human needed -- under the daily limit."""
    return {"decision": "auto-approved", "log": ["auto_process: approved, under daily limit"]}


def notify_node(state: TriageState) -> dict:
    """Node 4. Both branches end here -- the customer gets told the outcome either way."""
    return {"log": [f"notify: customer told -- {state['decision']}"]}


builder = StateGraph(TriageState)
builder.add_node("triage", triage_node)
builder.add_node("policy_check", policy_check_node)
builder.add_node("human_approval", human_approval_node)
builder.add_node("auto_process", auto_process_node)
builder.add_node("notify", notify_node)
builder.add_edge(START, "triage")
builder.add_edge("triage", "policy_check")
builder.add_conditional_edges(
    "policy_check", route_by_risk,
    {"human_approval": "human_approval", "auto_process": "auto_process"},
)
builder.add_edge("human_approval", "notify")
builder.add_edge("auto_process", "notify")
builder.add_edge("notify", END)

graph_no_checkpoint = builder.compile()  # deliberately no checkpointer yet
print("Graph compiled -- no checkpointer.")

# A SMALL request -- stays under the daily limit, should auto-process end to end.
small_result = graph_no_checkpoint.invoke(
    {"customer_message": "Please send 30,000 rupees via UPI to my landlord.", "log": []},
    config={"configurable": {"thread_id": "task1-small"}},
)
print("=" * 72)
print("TASK 1 -- SMALL request (auto-process path)")
print("=" * 72)
for line in small_result["log"]:
    print(" ", line)
print("Decision:", small_result["decision"])

# A LARGE request -- breaches the daily limit, should route to human_approval and PAUSE.
print()
print("=" * 72)
print("TASK 1 -- LARGE request (human_approval path)")
print("=" * 72)
large_result = graph_no_checkpoint.invoke(
    {"customer_message": "Please send 1,25,000 rupees via UPI to my supplier today.", "log": []},
    config={"configurable": {"thread_id": "task1-large"}},
)
for line in large_result["log"]:
    print(" ", line)
if "__interrupt__" in large_result:
    print("PAUSED --", large_result["__interrupt__"][0].value["question"])

# Now try to actually resume it -- with NO checkpointer configured.
print()
print("Attempting to resume the paused run above, with no checkpointer...")
try:
    graph_no_checkpoint.invoke(
        Command(resume="approved"), config={"configurable": {"thread_id": "task1-large"}}
    )
except RuntimeError as e:
    print(f"FAILED, as expected: {e}")
    print("This is exactly why Concept C exists -- without a checkpointer, there is nothing to")
    print("resume FROM. Go back to the concept page for Concept C.")


Graph compiled -- no checkpointer.
TASK 1 -- SMALL request (auto-process path)
  triage: extracted intent=upi_transfer, amount=30000.0
  policy_check: fetched UPI daily limit policy
  auto_process: approved, under daily limit
  notify: customer told -- auto-approved
Decision: auto-approved

TASK 1 -- LARGE request (human_approval path)
  triage: extracted intent=upi_transfer, amount=125000.0
  policy_check: fetched UPI daily limit policy
PAUSED -- Approve this Rs 125,000 UPI transfer for Customer? It breaches today's Rs 100,000 limit.

Attempting to resume the paused run above, with no checkpointer...
FAILED, as expected: Cannot use Command(resume=...) without checkpointer
This is exactly why Concept C exists -- without a checkpointer, there is nothing to
resume FROM. Go back to the concept page for Concept C.


**Notice:** the small request went start to finish in one call. The large one paused
correctly -- but trying to resume it failed, with a real error, because nothing was saved
anywhere to resume from. Now go back to the concept page for **Concept C**.


## Concept C, on the page — then come back here for Task 2

Read **Concept C — A Save File, Written After Every Step**, then run the cell below: the
SAME graph, recompiled with a real `SqliteSaver`, paused, and then resumed from a **completely
fresh Python object** pointed at the same file — today's flagship "kill and restart" moment.


In [5]:
# TASK 2 -- attach a real, file-backed checkpointer. Then prove persistence for real: build a
# BRAND NEW graph object (simulating a fresh process) pointed at the same database file, and
# show it remembers the paused run.

DB_PATH = "m3_2_checkpoints.sqlite"

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
checkpointer = SqliteSaver(conn)
graph = builder.compile(checkpointer=checkpointer)

# A fresh, unique thread_id every time this cell runs -- exactly what a real system does for
# every new customer conversation. It also means re-running this cell (e.g. to watch it again)
# always starts a brand-new, clean conversation instead of colliding with an old one.
config = {"configurable": {"thread_id": f"task2-restart-demo-{uuid.uuid4().hex[:8]}"}}

print("=" * 72)
print("TASK 2a -- run the large request, WITH a checkpointer this time")
print("=" * 72)
result = graph.invoke(
    {"customer_message": "Please send 1,25,000 rupees via UPI to my supplier today.", "log": []},
    config=config,
)
for line in result["log"]:
    print(" ", line)
print("PAUSED --", result["__interrupt__"][0].value["question"])

# "Kill" this process's objects entirely -- close the connection, delete the graph/checkpointer.
conn.close()
del graph, checkpointer, conn

print()
print("=" * 72)
print("TASK 2b -- brand new connection, brand new graph object, SAME database file")
print("=" * 72)
fresh_conn = sqlite3.connect(DB_PATH, check_same_thread=False)
fresh_checkpointer = SqliteSaver(fresh_conn)
fresh_graph = builder.compile(checkpointer=fresh_checkpointer)

state = fresh_graph.get_state(config)
print("This graph object has never seen this thread_id before this line.")
print("Still paused at:", state.next)
print("State it recovered from disk:", {k: v for k, v in state.values.items() if k != "log"})

resumed = fresh_graph.invoke(Command(resume="approved"), config=config)
print()
print("Resumed successfully on the NEW object:")
for line in resumed["log"]:
    print(" ", line)
print("Final decision:", resumed["decision"])
print()
print("Nothing was re-run from the top. The save file is what made this possible.")


TASK 2a -- run the large request, WITH a checkpointer this time
  triage: extracted intent=upi_transfer, amount=125000.0
  policy_check: fetched UPI daily limit policy
PAUSED -- Approve this Rs 125,000 UPI transfer for Customer? It breaches today's Rs 100,000 limit.

TASK 2b -- brand new connection, brand new graph object, SAME database file
This graph object has never seen this thread_id before this line.
Still paused at: ('human_approval',)
State it recovered from disk: {'customer_message': 'Please send 1,25,000 rupees via UPI to my supplier today.', 'customer_name': 'Customer', 'account_type': 'current', 'intent': 'upi_transfer', 'amount': 125000.0, 'kb_note': '[16] UPI transaction limits per day. UPI allows up to one lakh rupees per day across all apps linked to your account. New UPI registrations are capped at five thousand rupees for the first twenty four hours.'}

Resumed successfully on the NEW object:
  triage: extracted intent=upi_transfer, amount=125000.0
  policy_check: fet

**Notice:** `fresh_graph` never ran `triage` or `policy_check` itself -- it loaded
their results from the checkpoint file. That's the entire idea of durable execution. Now go back
to the concept page for **Concept D** -- today's required exercise.


## Concept D, on the page — then come back here for Task 3 (required exercise)

Read **Concept D — Pause for a Human, Then Resume Correctly** and try the approve/reject demo,
then run the cell below: the clean, explicit version of today's Hands-on Exercise, with
intermediate state streamed as it happens, tested both ways (approve AND reject).


In [6]:
# TASK 3 -- REQUIRED EXERCISE: pause before the sensitive step, resume correctly, on two
# separate requests (one approved, one rejected) so both outcomes of the human's decision are
# proven, not just one.

def run_and_show_pause(thread_id, message):
    cfg = {"configurable": {"thread_id": thread_id}}
    print(f"--- Streaming intermediate state for thread '{thread_id}' ---")
    last_printed = None
    for chunk in fresh_graph.stream({"customer_message": message, "log": []}, config=cfg, stream_mode="values"):
        node_log = chunk.get("log", [])
        # stream_mode="values" emits one chunk per graph super-step, including the routing
        # step itself -- skip re-printing when the log hasn't actually grown since last chunk.
        if node_log and node_log[-1] != last_printed:
            print("  state update ->", node_log[-1])
            last_printed = node_log[-1]
    state = fresh_graph.get_state(cfg)
    assert state.next == ("human_approval",), f"Expected a pause at human_approval, got {state.next}"
    print(f"  PAUSED as expected -- next up: {state.next}")
    return cfg


print("=" * 72)
print("TASK 3a -- APPROVE case")
print("=" * 72)
approve_thread = f"task3-approve-{uuid.uuid4().hex[:8]}"  # fresh per run, see Task 2's note
approve_cfg = run_and_show_pause(approve_thread, "Send 1,50,000 rupees via UPI to my supplier.")
approve_result = fresh_graph.invoke(Command(resume="approved"), config=approve_cfg)
print("  Final decision:", approve_result["decision"])
assert approve_result["decision"] == "approved"

print()
print("=" * 72)
print("TASK 3b -- REJECT case (a different thread_id -- a separate customer request)")
print("=" * 72)
reject_thread = f"task3-reject-{uuid.uuid4().hex[:8]}"
reject_cfg = run_and_show_pause(reject_thread, "Send 2,00,000 rupees via UPI to a new payee.")
reject_result = fresh_graph.invoke(Command(resume="rejected"), config=reject_cfg)
print("  Final decision:", reject_result["decision"])
assert reject_result["decision"] == "rejected"

print()
print("Both outcomes of the human decision resumed correctly, from the same paused point.")


TASK 3a -- APPROVE case
--- Streaming intermediate state for thread 'task3-approve-3afcaa4a' ---
  state update -> triage: extracted intent=upi_transfer, amount=150000.0
  state update -> policy_check: fetched UPI daily limit policy
  PAUSED as expected -- next up: ('human_approval',)
  Final decision: approved

TASK 3b -- REJECT case (a different thread_id -- a separate customer request)
--- Streaming intermediate state for thread 'task3-reject-06f8afd3' ---
  state update -> triage: extracted intent=upi_transfer, amount=200000.0
  state update -> policy_check: fetched UPI daily limit policy
  PAUSED as expected -- next up: ('human_approval',)
  Final decision: rejected

Both outcomes of the human decision resumed correctly, from the same paused point.


**Notice:** `stream_mode="values"` let you watch `log` grow in real time, instead of
waiting for one final printed block -- that's the streaming idea from Concept D's aside. Now go
back to the concept page for **Concept E**.


## Concept E, on the page — then come back here for Task 4

Read **Concept E — Recover from a Crash**, then run the cell below: a real exception inside a
node, and a recovery that resumes from that exact node only.


In [7]:
# TASK 4 -- simulate a real node failure (not a human-triggered pause) and recover from it.
# notify_node will raise the FIRST time it's called, then succeed on the retry -- standing in
# for a flaky external notification service.
#
# _already_failed lives in THIS cell's own closure, fresh every time you run the cell -- so
# re-running this cell (e.g. to watch the failure again) reliably fails once and recovers once,
# every time, instead of depending on kernel-wide state from an earlier run.
_already_failed = {"flag": False}


def notify_node_flaky(state: TriageState) -> dict:
    if not _already_failed["flag"]:
        _already_failed["flag"] = True
        raise RuntimeError("Simulated: Meridian's SMS notification service is unreachable")
    return {"log": [f"notify: customer told -- {state['decision']} (after one retry)"]}


# Rebuild the graph with the flaky notify node, same checkpointer/db file.
crash_builder = StateGraph(TriageState)
crash_builder.add_node("triage", triage_node)
crash_builder.add_node("policy_check", policy_check_node)
crash_builder.add_node("human_approval", human_approval_node)
crash_builder.add_node("auto_process", auto_process_node)
crash_builder.add_node("notify", notify_node_flaky)
crash_builder.add_edge(START, "triage")
crash_builder.add_edge("triage", "policy_check")
crash_builder.add_conditional_edges(
    "policy_check", route_by_risk,
    {"human_approval": "human_approval", "auto_process": "auto_process"},
)
crash_builder.add_edge("human_approval", "notify")
crash_builder.add_edge("auto_process", "notify")
crash_builder.add_edge("notify", END)
crash_graph = crash_builder.compile(checkpointer=fresh_checkpointer)

crash_cfg = {"configurable": {"thread_id": f"task4-crash-demo-{uuid.uuid4().hex[:8]}"}}

print("=" * 72)
print("TASK 4a -- small request (auto-process path, no human wait) -- notify WILL fail")
print("=" * 72)
try:
    crash_graph.invoke(
        {"customer_message": "Send 10,000 rupees via UPI to my sister.", "log": []},
        config=crash_cfg,
    )
except RuntimeError as e:
    print(f"Node raised, as designed: {e}")
    print("triage, policy_check and auto_process already ran and are safely checkpointed.")

print()
print("=" * 72)
print("TASK 4b -- re-invoke on the SAME thread_id -- recovers, does not restart")
print("=" * 72)
recovered = crash_graph.invoke(None, config=crash_cfg)
for line in recovered["log"]:
    print(" ", line)
assert recovered["log"][0].startswith("triage"), "triage should already be in the log from before the crash"
assert sum(1 for l in recovered["log"] if l.startswith("triage")) == 1, "triage must NOT have re-run"
print()
print("triage appears exactly once in the log -- it was not re-run. Only the failed notify step was.")


TASK 4a -- small request (auto-process path, no human wait) -- notify WILL fail
Node raised, as designed: Simulated: Meridian's SMS notification service is unreachable
triage, policy_check and auto_process already ran and are safely checkpointed.

TASK 4b -- re-invoke on the SAME thread_id -- recovers, does not restart
  triage: extracted intent=upi_transfer, amount=10000.0
  policy_check: fetched UPI daily limit policy
  auto_process: approved, under daily limit
  notify: customer told -- auto-approved (after one retry)

triage appears exactly once in the log -- it was not re-run. Only the failed notify step was.


**Notice:** compare this to Concept D — a human choosing to pause and a node
crashing are handled by the exact same underlying mechanic: resume from the last good checkpoint.
Now go back to the concept page for **Concept F** (stretch).


## Concept F, on the page — then come back here for Tasks 5 and 6 (stretch / Diamond)

Read **Concept F — Rewind to Any Past Save Point, and Replay It Differently**, then run the
cells below: fork Task 3's approved conversation from the point right before the human decision,
and replay it as a rejection — without touching the original approved outcome.


In [8]:
# TASK 5 -- STRETCH: time travel. Fork Task 3a's APPROVED thread from the checkpoint right
# before human_approval ran, and replay it as REJECTED instead.
#
# The one subtlety flagged on the concept page: invoking Command(resume=...) directly on an old
# checkpoint's config reuses that task's ALREADY-RECORDED result (it would silently give back
# "approved" again). update_state(...) first creates a genuinely NEW checkpoint with no cached
# result, so the node actually re-runs with the new resume value. Verified below.

history = list(fresh_graph.get_state_history(approve_cfg))
print(f"{len(history)} checkpoints in the approved thread's history.")

fork_point = next(h for h in history if h.next == ("human_approval",))
print("Fork point located -- next up was:", fork_point.next)

fresh_checkpoint_cfg = fresh_graph.update_state(fork_point.config, {})
forked_result = fresh_graph.invoke(Command(resume="rejected"), config=fresh_checkpoint_cfg)

print()
print("New forked branch, replayed with REJECTED:")
print("  Decision:", forked_result["decision"])
assert forked_result["decision"] == "rejected"

# Confirm the ORIGINAL approved checkpoint is still exactly as it was -- retrievable by its own id.
original_state = fresh_graph.get_state(fork_point.config)
print()
print("Original checkpoint (by its own checkpoint_id), untouched:")
print("  next:", original_state.next, "-- still paused there, exactly as Task 3a left it")
print()
print("Two different outcomes now exist from the same starting point -- neither overwrote the other.")


6 checkpoints in the approved thread's history.
Fork point located -- next up was: ('human_approval',)

New forked branch, replayed with REJECTED:
  Decision: rejected

Original checkpoint (by its own checkpoint_id), untouched:
  next: ('human_approval',) -- still paused there, exactly as Task 3a left it

Two different outcomes now exist from the same starting point -- neither overwrote the other.


In [9]:
# TASK 6 -- save your results. This file is your Capstone Phase 3 artifact.

key_takeaway = """
KEY TAKEAWAY: A LangGraph workflow is a shared state notebook (TriageState) passed through
plain-function nodes, connected by edges -- including a conditional edge that branches on real
risk data (Task 1). A checkpointer turns that into something durable: kill the process, reopen
a fresh one, and it resumes exactly where it paused (Task 2). interrupt() and Command(resume=...)
turn a pause into a genuine human approval gate, both ways (Task 3). The same save-file mechanic
that makes a human's pause resumable also recovers a workflow after a real crash (Task 4). And
every checkpoint stays inspectable forever -- you can rewind to any past decision point and
replay it differently, without erasing what actually happened (Task 5).
"""
print(key_takeaway)

results = {
    "task1_small_decision": small_result["decision"],
    "task1_large_paused": "__interrupt__" in large_result,
    "task2_restart_final_decision": resumed["decision"],
    "task3_approve_decision": approve_result["decision"],
    "task3_reject_decision": reject_result["decision"],
    "task4_recovered_log": recovered["log"],
    "task5_forked_decision": forked_result["decision"],
    "task5_original_still_paused": original_state.next == ("human_approval",),
    "key_takeaway": key_takeaway,
}

with open("m3_2_langgraph_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved m3_2_langgraph_results.json --", len(json.dumps(results)), "bytes")
print("This is Capstone Phase 3's working artifact -- a durable, resumable, human-supervised triage workflow.")



KEY TAKEAWAY: A LangGraph workflow is a shared state notebook (TriageState) passed through
plain-function nodes, connected by edges -- including a conditional edge that branches on real
risk data (Task 1). A checkpointer turns that into something durable: kill the process, reopen
a fresh one, and it resumes exactly where it paused (Task 2). interrupt() and Command(resume=...)
turn a pause into a genuine human approval gate, both ways (Task 3). The same save-file mechanic
that makes a human's pause resumable also recovers a workflow after a real crash (Task 4). And
every checkpoint stays inspectable forever -- you can rewind to any past decision point and
replay it differently, without erasing what actually happened (Task 5).

Saved m3_2_langgraph_results.json -- 1262 bytes
This is Capstone Phase 3's working artifact -- a durable, resumable, human-supervised triage workflow.
